# Predicting code-switches and their duration (XLM-R vs. Qwen3)

**Data:** SwitchLingua_text (`first_language` x `second_language`), the pairs listed in `config.PAIRS` (enable needed languages there). All active pairs are trained together in one model; evaluation is reported per pair.

**Task:** one shared encoder with two heads
- **switch head** (binary): will the next unit change language?
- **duration head** (`N_DUR = 4` classes): the length class of the upcoming segment.
  - `0` = no switch
  - `1` = the new-language segment lasts 1-2 units
  - `2` = 3-7 units
  - `3` = >= 8 units

  Class `0` is trained with the rest, so the head also sees no-switch positions. At evaluation and inference the duration is read only from classes `1..3` (`argmax` over those logits), and `dur_acc` / `dur_MAE` are computed only where the gold label is a real switch (`dur > 0`). The ordinal L1 penalty is applied only to switch positions and only over classes `1..3`, because class `0` is not "shorter" than class `1`.

**LID labels** (0 = non-English L1, 1 = English) are only used to *generate* the labels (kept as `last_lang` / `next_lang` for inspection); they are never model inputs. Script decides LID (Han / Devanagari / ... -> 0, Latin -> 1); Latin-script pairs (and **(TO-DO)** `hi-en`, which also covers romanized Hindi) use wordfreq dictionaries per word, with ambiguous words resolved from the nearest determined neighbours.

**Units:** Han characters and kana = one character; all other scripts = one word. Context = everything from the start of the sentence to the current position, identically in training and inference.

> Duration bins count units, which are not fully comparable across pairs (characters vs. words), so always read results per pair. `dur_MAE` is measured in bins (classes 1-3, so each error is at most 2), not in units; prefer `dur_acc` when comparing.

## Project layout
```
pipeline.ipynb          <- this notebook: pipeline, results, commentary
codeswitch/
  config.py             hyperparameters, language pairs, paths
  utils.py              device, seeding, memory monitoring, class weights
  lid.py                unit segmentation + language identification
  data.py               corpus loading, split by conversation, example generation
  metrics.py            score / per-pair score / result tables
  baselines.py          B0 prior, B1 lookup table, B2 char n-gram LR
  trainer.py            shared training loop (differential LR, optional best-epoch restore)
  xlmr.py               XLM-R dataset, model, evaluate, predict
  qwen.py               Qwen3 dataset, model, evaluate, predict
```
Sections 4-5 are the XLM-R pipeline; section 6 is Qwen3. Both use the same train/val sentences, the same head definitions and the same `score` function, so results are directly comparable. Change hyperparameters in `codeswitch/config.py` (picked up by `%autoreload`).

## 0. Setup

In [1]:
# !pip install -q ipykernel transformers torch numpy pandas scikit-learn wordfreq psutil
%load_ext autoreload
%autoreload 2

import numpy as np
from codeswitch import config as C
from codeswitch.utils import DEVICE, set_seed, mem, free
from codeswitch import lid, data, baselines, xlmr, qwen
from codeswitch.metrics import results_table, save_results, print_scores

set_seed(C.SEED)
print("device:", DEVICE)
results = {}      # {model name: {pair or "ALL": metrics}}

/Users/WanderingCorn/Documents/Northeastern/CS6140/codeswitch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. Segmentation + LID self-test
One example sentence per pair, printed as `(unit, label)`; `0` = L1, `1` = English.

In [2]:
lid.selftest()

zh-en [('我', 0), ('喜', 0), ('欢', 0), ('this', 1), ('movie', 1), ('因', 0), ('为', 0), ("it's", 1), ('很', 0), ('好', 0), ('看', 0)]
yue-en [('佢', 0), ('今', 0), ('日', 0), ('meeting', 1), ('完', 0), ('咗', 0), ('先', 0), ('返', 0), ('屋', 0), ('企', 0)]
hi-en [('मुझे', 0), ('यह', 0), ('movie', 1), ('बहुत', 0), ('पसंद', 0), ('आई', 0), ('because', 1), ('it', 1), ('was', 1), ('amazing', 1)]
hi-en [('mujhe', 1), ('yeh', 1), ('movie', 1), ('bahut', 1), ('pasand', 1), ('aayi', 1), ('because', 1), ('it', 1), ('was', 1), ('amazing', 1)]
es-en [('Ayer', 0), ('fui', 0), ('al', 0), ('cine', 0), ('con', 0), ('mi', 0), ('amigo', 0), ('and', 1), ('we', 1), ('watched', 1), ('a', 1), ('really', 1), ('good', 1), ('movie', 1), ('pero', 0), ('no', 0), ('me', 0), ('gustó', 0), ('el', 0), ('final', 0)]
ar-en [('أنا', 0), ('أحب', 0), ('هذا', 0), ('movie', 1), ('كثيراً', 0), ('because', 1), ("it's", 1), ('great', 1)]
ja-en [('今', 0), ('日', 0), ('は', 0), ('meeting', 1), ('が', 0), ('あ', 0), ('る', 0), ('の', 0), ('で', 0), ("

## 2. Load the corpus
Each CSV row is a conversation; `data_generation_result` is a stringified list of sentences. Sentences are expanded to records tagged with `pair` and `conv` so the split can be done per conversation. The health check flags pairs whose L1 share is < 5% (LID not applicable, e.g. romanized text).

In [3]:
records = data.load_records()
data.corpus_report(records)

first_language  second_language
Cantonese       English            119248
Arabic          English              8548
Philippines     English              8537
German          French               8511
Chinese         English              8458
Vietnamese      English              8457
Malay           English              8386
Japanese        English              8385
Hindi           English              8381
Korean          English              8266
Spanish         English              8110
French          English              8070
Russian         English              8017
Italian         English              7979
German          English              6819

11327 sentences; 1 conversations failed to parse -> parse_failed_conversations.txt
hi-en   sentences   3236 | L1 units    13169 | EN units    43384 | sentences with a switch 38.3%
yue-en  sentences   5039 | L1 units    82224 | EN units    18475 | sentences with a switch 69.8%
es-en   sentences   3052 | L1 units    36364 | EN units    1

## 3. Split dataset split and build example
**Dataset split:** Training and validation dataset are split by conversations. Sentences from the same conversation cannot be assigned to differentdatasets.

**Example:** Every stopping point (after a unit, or after each punctuation mark / number) becomes one example: `text` = prefix, `switch` = does the next unit change language, `duration` = length class of the new segment (0 if no switch).

**Variable naming:** rec - conversation, raw - prefix text,

In [4]:
train_recs, val_recs = data.split_records(records)
train_raw = data.build_examples(train_recs)
val_raw = data.build_examples(val_recs)

for e in data.make_examples("मुझे यह movie बहुत पसंद आई, because it's amazing।", "hi-en")[:8]:
    print(e)
print()
data.example_report(train_raw, val_raw)

{'text': 'मुझे', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'L1', 'next_lang': 'L1'}
{'text': 'मुझे यह', 'pair': 'hi-en', 'switch': 1, 'duration': 1, 'last_lang': 'L1', 'next_lang': 'en'}
{'text': 'मुझे यह movie', 'pair': 'hi-en', 'switch': 1, 'duration': 2, 'last_lang': 'en', 'next_lang': 'L1'}
{'text': 'मुझे यह movie बहुत', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'L1', 'next_lang': 'L1'}
{'text': 'मुझे यह movie बहुत पसंद', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'L1', 'next_lang': 'L1'}
{'text': 'मुझे यह movie बहुत पसंद आई', 'pair': 'hi-en', 'switch': 1, 'duration': 2, 'last_lang': 'L1', 'next_lang': 'en'}
{'text': 'मुझे यह movie बहुत पसंद आई,', 'pair': 'hi-en', 'switch': 1, 'duration': 2, 'last_lang': 'L1', 'next_lang': 'en'}
{'text': 'मुझे यह movie बहुत पसंद आई, because', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'en', 'next_lang': 'en'}

train: 198006 examples, switch=1 share 9.5%
duration class counts: [179124, 72

## 4. Naive baselines (each pair trained and evaluated separately)

- **B0 prior:** always / never predict a switch; duration = most common class (lower bound)
- **B1 lookup:** `P(switch | current language, run length, trailing punctuation)`; duration = mode per language. No text representation at all
- **B2 char n-gram + LR:** TF-IDF on the last 6 characters. Surface form, no pretraining

If XLM-R cannot beat B1, the signal is essentially run length + punctuation; if it cannot beat B2, pretraining adds nothing.

In [5]:
results.update(baselines.run_baselines(train_raw, val_raw))

for metric in ["switch_F1","switch_P", "switch_R"]:
    print(f"\n== {metric} ==")
    display(results_table(results, metric))


== switch_F1 ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
hi-en,0.138,0.0,0.310,0.388
yue-en,0.172,0.0,0.417,0.350
es-en,0.213,0.0,0.361,0.469



== switch_P ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
hi-en,0.074,0.0,0.185,0.261
yue-en,0.094,0.0,0.349,0.247
es-en,0.119,0.0,0.233,0.342



== switch_R ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
hi-en,1.0,0.0,0.946,0.757
yue-en,1.0,0.0,0.519,0.599
es-en,1.0,0.0,0.811,0.747


## 5. XLM-R (shared encoder + two heads on the `<mask>` position)

Training uses **negative downsampling** (keep all `switch=1`, a `KEEP_NEG` fraction of `switch=0`; class weights computed on the downsampled set) and **length-bucketed batches** to reduce padding. Validation is never downsampled.

In [6]:
free(); mem("before XLM-R")
xlmr_tok, xlmr_model, xlmr_train_dl, xlmr_val_dl = xlmr.build(train_raw, val_raw)
mem("XLM-R loaded")

[mem before XLM-R] MPS alloc 0.0 GB | driver 0.0 GB | RSS 3.3 GB
downsampling: 198006 -> 72615, switch=1 share 26.0%
switch class weights: [0.8220109939575195, 1.3866733312606812]


W1021 11:23:26.437000 17207 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8624.82it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[mem XLM-R loaded] MPS alloc 1.1 GB | driver 1.1 GB | RSS 3.9 GB


In [7]:
history = xlmr.train(xlmr_model, xlmr_train_dl, xlmr_val_dl)
results["XLM-R"] = xlmr.evaluate(xlmr_model, xlmr_val_dl)

[mem XLM-R step 0] MPS alloc 3.4 GB | driver 6.4 GB | RSS 4.0 GB
epoch 1: train loss 2.3718
  val [ALL   ] {'switch_P': 0.4, 'switch_R': 0.85, 'switch_F1': 0.544, 'dur_acc': 0.548, 'dur_MAE': 0.542, 'val_loss': np.float64(1.2602), 'val_loss_cs': np.float64(0.2988), 'val_loss_dur': np.float64(0.9614)}
  val [hi-en ] {'switch_P': 0.398, 'switch_R': 0.946, 'switch_F1': 0.561, 'dur_acc': 0.568, 'dur_MAE': 0.516}
  val [yue-en] {'switch_P': 0.373, 'switch_R': 0.791, 'switch_F1': 0.507, 'dur_acc': 0.602, 'dur_MAE': 0.441}
  val [es-en ] {'switch_P': 0.442, 'switch_R': 0.874, 'switch_F1': 0.587, 'dur_acc': 0.458, 'dur_MAE': 0.702}
  ^ best so far
[mem XLM-R epoch 1] MPS alloc 3.4 GB | driver 3.5 GB | RSS 2.5 GB
epoch 2: train loss 1.8044
  val [ALL   ] {'switch_P': 0.506, 'switch_R': 0.791, 'switch_F1': 0.617, 'dur_acc': 0.594, 'dur_MAE': 0.501, 'val_loss': np.float64(1.0818), 'val_loss_cs': np.float64(0.2396), 'val_loss_dur': np.float64(0.8421)}
  val [hi-en ] {'switch_P': 0.549, 'switch_R':

### XLM-R inference on partially typed text

In [8]:
probes = ["我喜欢", "我喜欢 this", "I think 这个", "今天 weather 不错，",
          "मुझे यह", "मुझे यह movie", "佢今日 meeting", "聽講DeepSeek的AI"]
for s in probes:
    print(f"{s!r:25} {xlmr.predict(xlmr_model, xlmr_tok, s)}")

'我喜欢'                     {'p_switch': 0.195, 'duration_if_switch': 1, 'p_duration': [0.856, 0.123, 0.02, 0.001]}
'我喜欢 this'                {'p_switch': 0.45, 'duration_if_switch': 1, 'p_duration': [0.717, 0.16, 0.11, 0.013]}
'I think 这个'              {'p_switch': 0.312, 'duration_if_switch': 1, 'p_duration': [0.736, 0.251, 0.013, 0.0]}
'今天 weather 不错，'          {'p_switch': 0.722, 'duration_if_switch': 2, 'p_duration': [0.445, 0.221, 0.267, 0.066]}
'मुझे यह'                 {'p_switch': 0.165, 'duration_if_switch': 1, 'p_duration': [0.849, 0.143, 0.008, 0.0]}
'मुझे यह movie'           {'p_switch': 0.933, 'duration_if_switch': 1, 'p_duration': [0.103, 0.651, 0.229, 0.016]}
'佢今日 meeting'             {'p_switch': 0.987, 'duration_if_switch': 2, 'p_duration': [0.032, 0.358, 0.463, 0.147]}
'聽講DeepSeek的AI'           {'p_switch': 0.989, 'duration_if_switch': 3, 'p_duration': [0.016, 0.108, 0.172, 0.704]}


In [9]:
# Free memory before loading Qwen
del xlmr_model, xlmr_train_dl, xlmr_val_dl
free(); mem("after XLM-R cleanup")

[mem after XLM-R cleanup] MPS alloc 0.0 GB | driver 0.0 GB | RSS 1.4 GB


## 6. Qwen3 (decoder-only)

Differences from the XLM-R version:
- **One sentence per sample:** the causal mask lets one forward pass produce predictions for all stopping points
- **Labels on the last token of each chunk** (a decoder only knows what a unit is after reading all of it); other positions are `-100`
- **Chunked tokenization:** the sentence is split at stopping points and each chunk tokenized separately, so stopping points always fall on token boundaries
- **Heads on every position's hidden state;** the loss ignores `-100`
- Embedding is frozen (memory + overfitting); duration loss adds label smoothing and an ordinal L1 term
- Labelled positions match the XLM-R examples one-to-one (asserted in `check_alignment`)

In [10]:
free(); mem("before Qwen")
qwen_tok, qwen_model, qwen_train_dl, qwen_val_dl, qwen_train_ds = qwen.build(train_recs, val_recs)
qwen.show_alignment(train_recs, qwen_tok)
mem("Qwen loaded")

[mem before Qwen] MPS alloc 0.0 GB | driver 0.0 GB | RSS 1.4 GB
0 / 10196 training sentences exceed QWEN_MAX_LEN=256 tokens (tail labels dropped)
switch class weights: [0.7434422969818115, 2.289813756942749]


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6138.36it/s]


hi-en | Maruti Brezza के नए features बहुत impressive हैं, you know?
'Mar'          switch=-100 dur=-100
'uti'          switch=   0 dur=   0
'ĠBre'         switch=-100 dur=-100
'zza'          switch=   1 dur=   1
'Ġà¤ķ'         switch=-100 dur=-100
'à¥ĩ'          switch=   0 dur=   0
'Ġà¤'          switch=-100 dur=-100
'¨'            switch=-100 dur=-100
'à¤ı'          switch=   1 dur=   1
'Ġfeatures'    switch=   1 dur=   1
'Ġà¤'          switch=-100 dur=-100
'¬'            switch=-100 dur=-100
'à¤¹'          switch=-100 dur=-100
'à¥ģ'          switch=-100 dur=-100
'à¤¤'          switch=   1 dur=   1
'Ġimpressive'  switch=   1 dur=   1
'Ġà¤¹'         switch=-100 dur=-100
'à¥'           switch=-100 dur=-100
'Ī'            switch=-100 dur=-100
'à¤Ĥ'          switch=   1 dur=   1
','            switch=   1 dur=   1
'Ġyou'         switch=   0 dur=   0
'Ġknow'        switch=-100 dur=-100
'?'            switch=-100 dur=-100
[mem Qwen loaded] MPS alloc 2.4 GB | driver 2.8 GB | RSS 2.0 GB


In [11]:
history_q = qwen.train(qwen_model, qwen_train_dl, qwen_val_dl)   # restores the best epoch
results["Qwen3"] = qwen.evaluate(qwen_model, qwen_val_dl)

[mem Qwen step 0] MPS alloc 5.9 GB | driver 11.6 GB | RSS 2.0 GB
epoch 1: train loss 2.5960
  val [ALL   ] {'switch_P': 0.647, 'switch_R': 0.669, 'switch_F1': 0.658, 'dur_acc': 0.58, 'dur_MAE': 0.539, 'dur_MAE_median': 0.503, 'val_loss': np.float64(2.1228), 'val_loss_cs': np.float64(0.26), 'val_loss_dur': np.float64(1.8628)}
  val [hi-en ] {'switch_P': 0.621, 'switch_R': 0.787, 'switch_F1': 0.694, 'dur_acc': 0.558, 'dur_MAE': 0.603}
  val [yue-en] {'switch_P': 0.668, 'switch_R': 0.553, 'switch_F1': 0.605, 'dur_acc': 0.588, 'dur_MAE': 0.488}
  val [es-en ] {'switch_P': 0.644, 'switch_R': 0.761, 'switch_F1': 0.698, 'dur_acc': 0.581, 'dur_MAE': 0.573}
  ^ best so far
[mem Qwen epoch 1] MPS alloc 5.9 GB | driver 7.2 GB | RSS 4.2 GB
epoch 2: train loss 1.9901
  val [ALL   ] {'switch_P': 0.643, 'switch_R': 0.674, 'switch_F1': 0.658, 'dur_acc': 0.559, 'dur_MAE': 0.497, 'dur_MAE_median': 0.5, 'val_loss': np.float64(2.1009), 'val_loss_cs': np.float64(0.2579), 'val_loss_dur': np.float64(1.843)}


### Qwen3 inference on partially typed text

In [12]:
for s in probes:
    print(f"{s!r:25} Qwen3: {qwen.predict(qwen_model, qwen_tok, s)}")

'我喜欢'                     Qwen3: {'p_switch': 0.117, 'duration_if_switch': 1, 'p_duration': [0.892, 0.058, 0.03, 0.021]}
'我喜欢 this'                Qwen3: {'p_switch': 0.057, 'duration_if_switch': 2, 'p_duration': [0.891, 0.029, 0.04, 0.039]}
'I think 这个'              Qwen3: {'p_switch': 0.494, 'duration_if_switch': 1, 'p_duration': [0.766, 0.186, 0.035, 0.013]}
'今天 weather 不错，'          Qwen3: {'p_switch': 0.418, 'duration_if_switch': 2, 'p_duration': [0.758, 0.093, 0.106, 0.043]}
'मुझे यह'                 Qwen3: {'p_switch': 0.147, 'duration_if_switch': 1, 'p_duration': [0.886, 0.071, 0.025, 0.017]}
'मुझे यह movie'           Qwen3: {'p_switch': 0.774, 'duration_if_switch': 1, 'p_duration': [0.481, 0.326, 0.15, 0.043]}
'佢今日 meeting'             Qwen3: {'p_switch': 0.937, 'duration_if_switch': 2, 'p_duration': [0.224, 0.143, 0.44, 0.193]}
'聽講DeepSeek的AI'           Qwen3: {'p_switch': 0.92, 'duration_if_switch': 3, 'p_duration': [0.302, 0.082, 0.2, 0.415]}


## 7. Comparison
Rows = language pairs (`ALL` = pooled, only for the jointly trained models); baselines are trained per pair.

In [13]:
import pandas as pd
for metric in ["switch_F1"]: #"switch_P", "switch_R", "dur_acc", "dur_MAE"
    print(f"\n== {metric} ==")
    display(results_table(results, metric))
save_results(results, "results.json")


== switch_F1 ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR,XLM-R,Qwen3
ALL,NaN,NaN,NaN,NaN,0.617,0.658
hi-en,0.138,0.0,0.310,0.388,0.673,0.697
yue-en,0.172,0.0,0.417,0.350,0.569,0.606
es-en,0.213,0.0,0.361,0.469,0.651,0.700
